# Turkish Legal RAG - Reranker Fine-Tuning

Bu notebook `reranker.jsonl` dosyasını kullanarak BERTurk tabanlı cross-encoder reranker fine-tune eder ve sonra fine-tuned modeli mevcut RAG retrieval pipeline içinde test eder.

Kaggle ayarları: GPU açık olmalı, Internet açık olmalı.

In [ ]:
!nvidia-smi
!echo "Input files:"
!find /kaggle/input -maxdepth 5 -type f | sort | head -200

## 1. Paket Kurulumu

In [ ]:
!pip install -q -U sentence-transformers faiss-cpu

## 2. Dosyaları Çalışma Klasörüne Kopyala

Dataset path farklıysa `INPUT_CANDIDATES` listesine Kaggle'da görünen path'i ekleyin.

In [ ]:
from pathlib import Path
import shutil

INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/efealvs/turkish-legal-rag-data"),
    Path("/kaggle/input/turkish-legal-rag-data"),
]

INPUT_DIR = next((p for p in INPUT_CANDIDATES if p.exists()), None)
if INPUT_DIR is None:
    print("Known input paths not found. Available folders:")
    for p in Path("/kaggle/input").glob("**/*"):
        if p.is_dir():
            print(p)
    raise FileNotFoundError("Kaggle dataset folder not found.")

WORK_DIR = Path("/kaggle/working/legal-rag")
for d in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/reranker",
    WORK_DIR / "data/eval",
    WORK_DIR / "data/index",
    WORK_DIR / "data/processed",
    WORK_DIR / "models",
]:
    d.mkdir(parents=True, exist_ok=True)

def copy_required(name: str, target_dir: Path):
    source = INPUT_DIR / name
    if not source.exists():
        raise FileNotFoundError(f"Missing required file in Kaggle dataset: {source}")
    target = target_dir / name
    shutil.copy2(source, target)
    print(f"Copied {name} -> {target}")

for name in ["split_reranker_jsonl.py", "train_reranker.py", "rerank_search.py", "mine_hard_negatives.py", "evaluate_retrieval.py"]:
    copy_required(name, WORK_DIR / "scripts")

for name in ["reranker.jsonl", "curated_queries.jsonl"]:
    copy_required(name, WORK_DIR / "data/reranker")

for name in ["qa_benchmark_gold.csv"]:
    copy_required(name, WORK_DIR / "data/eval")

for name in ["faiss_bge_m3.index", "metadata_bge_m3.json", "index_config_bge_m3.json"]:
    copy_required(name, WORK_DIR / "data/index")

for name in ["retrieval_corpus.json", "retrieval_chunks.json"]:
    copy_required(name, WORK_DIR / "data/processed")

print("\nPrepared files:")
!find /kaggle/working/legal-rag -maxdepth 4 -type f | sort

## 3. Reranker Dataset Kontrolü

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

reranker_path = Path("/kaggle/working/legal-rag/data/reranker/reranker.jsonl")
rows = [json.loads(line) for line in reranker_path.open(encoding="utf-8") if line.strip()]
groups = defaultdict(list)
for row in rows:
    groups[row.get("query_id") or row.get("query")].append(row)

print("Rows:", len(rows))
print("Query groups:", len(groups))
print("Labels:", Counter(row["label"] for row in rows))
print("Sources:", Counter(row.get("source") for row in rows))
print("Sample:")
print(json.dumps(rows[0], ensure_ascii=False, indent=2)[:1200])

## 4. Hard Negative Mining + Train/Dev Split

Önce mevcut `reranker.jsonl` üstüne bizim gerçek retrieval corpus'tan hard negative örnekleri ekliyoruz. Sonra split `query_id` bazlı yapılır; böylece aynı soru hem train hem dev içine girmez.

In [ ]:
!python /kaggle/working/legal-rag/scripts/mine_hard_negatives.py \
  --input /kaggle/working/legal-rag/data/reranker/reranker.jsonl \
  --output /kaggle/working/legal-rag/data/reranker/reranker_augmented.jsonl \
  --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json \
  --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index \
  --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json \
  --seed-queries /kaggle/working/legal-rag/data/reranker/curated_queries.jsonl \
  --stats-out /kaggle/working/legal-rag/data/reranker/reranker_augmented.stats.json \
  --embedding-device cuda \
  --embedding-batch-size 16 \
  --negatives-per-query 3 \
  --preliminary-top-k 80

!python /kaggle/working/legal-rag/scripts/split_reranker_jsonl.py \
  --input /kaggle/working/legal-rag/data/reranker/reranker_augmented.jsonl \
  --train-out /kaggle/working/legal-rag/data/reranker/train.jsonl \
  --dev-out /kaggle/working/legal-rag/data/reranker/dev.jsonl \
  --dev-ratio 0.1

!cat /kaggle/working/legal-rag/data/reranker/reranker_augmented.stats.json

## 5. BERTurk Cross-Encoder Reranker Fine-Tuning

Bu sürümde augmented dataset ile `epochs=2` deniyoruz ve dev skoruna göre en iyi modeli saklıyoruz. GPU memory hatası olursa `--batch-size 4` yapın.

In [ ]:
!rm -rf /kaggle/working/legal-rag/models/legal-berturk-reranker

!python /kaggle/working/legal-rag/scripts/train_reranker.py \
  --train /kaggle/working/legal-rag/data/reranker/train.jsonl \
  --dev /kaggle/working/legal-rag/data/reranker/dev.jsonl \
  --base-model dbmdz/bert-base-turkish-cased \
  --output-dir /kaggle/working/legal-rag/models/legal-berturk-reranker \
  --epochs 2 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --evaluation-steps 200 \
  --save-best-model \
  --device cuda

## 6. Model Dosyalarını Kontrol Et

In [ ]:
!ls -lh /kaggle/working/legal-rag/models/legal-berturk-reranker
!test -f /kaggle/working/legal-rag/models/legal-berturk-reranker/model.safetensors || test -f /kaggle/working/legal-rag/models/legal-berturk-reranker/pytorch_model.bin

## 7. Fine-Tuned Reranker Testleri

In [ ]:
!python /kaggle/working/legal-rag/scripts/rerank_search.py \
  "birini öldürmek suç mudur?" \
  --top-k 5 \
  --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index \
  --metadata /kaggle/working/legal-rag/data/index/metadata_bge_m3.json \
  --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json \
  --articles /kaggle/working/legal-rag/data/processed/retrieval_corpus.json \
  --reranker-model /kaggle/working/legal-rag/models/legal-berturk-reranker \
  --ranking-mode rerank \
  --device cuda \
  --rerank-batch-size 8 \
  --show-text

In [ ]:
!python /kaggle/working/legal-rag/scripts/rerank_search.py \
  "işçi 2 gün işe gelmezse ne olur?" \
  --top-k 5 \
  --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index \
  --metadata /kaggle/working/legal-rag/data/index/metadata_bge_m3.json \
  --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json \
  --articles /kaggle/working/legal-rag/data/processed/retrieval_corpus.json \
  --reranker-model /kaggle/working/legal-rag/models/legal-berturk-reranker \
  --ranking-mode rerank \
  --device cuda \
  --rerank-batch-size 8 \
  --show-text

In [ ]:
!python /kaggle/working/legal-rag/scripts/rerank_search.py \
  "kişisel veriler yurt dışına hangi şartlarda aktarılır?" \
  --top-k 5 \
  --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index \
  --metadata /kaggle/working/legal-rag/data/index/metadata_bge_m3.json \
  --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json \
  --articles /kaggle/working/legal-rag/data/processed/retrieval_corpus.json \
  --reranker-model /kaggle/working/legal-rag/models/legal-berturk-reranker \
  --ranking-mode rerank \
  --device cuda \
  --rerank-batch-size 8 \
  --show-text

## 8. Output Olarak Saklanacaklar

Kaggle notebook bitince `/kaggle/working/legal-rag/models/legal-berturk-reranker` klasörünü output olarak saklayın. Bu klasör bizim fine-tuned reranker modelimizdir.